# PERSUADE Analysis v6: Final Controls & Interaction Tests

**Goal:** Lock in the conclusion that score-bin differences in curve shape are explained by baseline fluency.

**Additional Analyses:**
1. Model B'/C' using ppl_W64 (robustness check)
2. Interaction test: does fluency→AUC slope differ by group?
3. Clear visualizations showing effect collapse
4. Findings summary for writeup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.integrate import trapezoid
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from pathlib import Path
from collections import Counter
import re

In [ ]:
# Mount Google Drive (for Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path("/content/drive/MyDrive/LRTIA/Results/Persuade")
    DATA_DIR = Path("/content/drive/MyDrive/LRTIA/Data/persuade_clean/cohorts")
    IN_COLAB = True
except:
    BASE_DIR = Path("../results/persuade")
    DATA_DIR = Path("../data/persuade_clean/cohorts")
    IN_COLAB = False

print(f"Results dir: {BASE_DIR}")

In [ ]:
# Configuration
EXPERIMENT = 'score_long'  # 'score' or 'score_long'
WINDOWS_PRIMARY = [32, 64, 128]
PRIMARY_METRICS = ['half_life_128', 'auc_128', 'log_slope_128', 'delta_128', 'slope_early_128', 'slope_late_128']
KEY_METRICS = ['auc_128', 'log_slope_128']  # Focus metrics for interaction tests

COLORS = {'low': '#e74c3c', 'mid': '#f39c12', 'high': '#2ecc71'}
GROUP_ORDER = ['low', 'mid', 'high']

print(f"Experiment: {EXPERIMENT}")

In [ ]:
# Load data
import json

results_path = BASE_DIR / EXPERIMENT / 'essay_level_results.csv'
df = pd.read_csv(results_path)
print(f"Loaded {len(df)} essays")

# Load text for repetition metrics
cohort_file = f"persuade_{EXPERIMENT.replace('_long', '_long_')}_cohort.jsonl" if 'long' in EXPERIMENT else f"persuade_{EXPERIMENT}_cohort.jsonl"
cohort_file = cohort_file.replace('_long_', '_long_')
if EXPERIMENT == 'score_long':
    cohort_file = 'persuade_score_long_cohort.jsonl'
elif EXPERIMENT == 'score':
    cohort_file = 'persuade_score_cohort.jsonl'

cohort_path = DATA_DIR / cohort_file
cohort_texts = {}
try:
    with open(cohort_path, 'r') as f:
        for line in f:
            r = json.loads(line)
            cohort_texts[r['essay_id']] = r['text']
    print(f"Loaded {len(cohort_texts)} essay texts")
except:
    print(f"Could not load cohort texts from {cohort_path}")

print(f"\nScore bin distribution:")
print(df['score_bin'].value_counts())

## 1. Compute Metrics (if not already present)

In [ ]:
# Compute *_128 metrics if not present
def compute_half_life(ppl_dict, percentile=0.5):
    if len(ppl_dict) < 2:
        return np.nan
    items = sorted(ppl_dict.items())
    windows = np.array([x[0] for x in items])
    ppls = np.array([x[1] for x in items])
    total_benefit = ppls[0] - ppls[-1]
    if total_benefit <= 0:
        return np.nan
    target_ppl = ppls[0] - percentile * total_benefit
    for i in range(len(ppls) - 1):
        if ppls[i] >= target_ppl >= ppls[i + 1]:
            frac = (ppls[i] - target_ppl) / (ppls[i] - ppls[i + 1])
            return windows[i] + frac * (windows[i + 1] - windows[i])
    return windows[-1]

def compute_primary_metrics(row):
    ppls = {}
    for W in WINDOWS_PRIMARY:
        col = f'ppl_W{W}'
        if col in row and pd.notna(row[col]):
            ppls[W] = row[col]
    
    result = {m: np.nan for m in PRIMARY_METRICS}
    if len(ppls) < 3:
        return pd.Series(result)
    
    ppl_32, ppl_64, ppl_128 = ppls[32], ppls[64], ppls[128]
    result['half_life_128'] = compute_half_life(ppls, 0.5)
    result['delta_128'] = ppl_32 - ppl_128
    
    windows = np.array([32, 64, 128])
    delta_ppl = np.array([0, ppl_32 - ppl_64, ppl_32 - ppl_128])
    result['auc_128'] = trapezoid(delta_ppl, windows)
    
    slope, _, _, _, _ = stats.linregress(np.log(windows), delta_ppl)
    result['log_slope_128'] = slope
    result['slope_early_128'] = (ppl_32 - ppl_64) / (np.log(64) - np.log(32))
    result['slope_late_128'] = (ppl_64 - ppl_128) / (np.log(128) - np.log(64))
    
    return pd.Series(result)

# Check if metrics exist, compute if not
if 'auc_128' not in df.columns:
    print("Computing *_128 metrics...")
    metrics_128 = df.apply(compute_primary_metrics, axis=1)
    for col in metrics_128.columns:
        df[col] = metrics_128[col]
else:
    print("*_128 metrics already present")

# Compute repetition metrics if not present
def tokenize_words(text):
    return re.findall(r'\b[a-z]+\b', text.lower())

def compute_repetition_metrics(text):
    words = tokenize_words(text)
    n_words = len(words)
    if n_words < 3:
        return {'ttr': np.nan, 'repeat_bigram_rate': np.nan, 'repeat_trigram_rate': np.nan, 'distinct2': np.nan, 'distinct3': np.nan}
    
    ttr = len(set(words)) / n_words
    bigrams = [tuple(words[i:i+2]) for i in range(len(words)-1)]
    trigrams = [tuple(words[i:i+3]) for i in range(len(words)-2)]
    bigram_counts = Counter(bigrams)
    trigram_counts = Counter(trigrams)
    
    return {
        'ttr': ttr,
        'repeat_bigram_rate': sum(1 for bg in bigrams if bigram_counts[bg] > 1) / len(bigrams) if bigrams else np.nan,
        'repeat_trigram_rate': sum(1 for tg in trigrams if trigram_counts[tg] > 1) / len(trigrams) if trigrams else np.nan,
        'distinct2': len(bigram_counts) / len(bigrams) if bigrams else np.nan,
        'distinct3': len(trigram_counts) / len(trigrams) if trigrams else np.nan,
    }

if 'ttr' not in df.columns and cohort_texts:
    print("Computing repetition metrics...")
    rep_data = []
    for _, row in df.iterrows():
        text = cohort_texts.get(row['essay_id'], '')
        m = compute_repetition_metrics(text)
        m['essay_id'] = row['essay_id']
        rep_data.append(m)
    df_rep = pd.DataFrame(rep_data)
    df = df.merge(df_rep, on='essay_id', how='left')
else:
    print("Repetition metrics already present or no text available")

print(f"\nColumns: {list(df.columns)}")

In [ ]:
# Standardize variables
def standardize(s):
    return (s - s.mean()) / s.std()

df['token_count_z'] = standardize(df['token_count'])
df['ppl_W32_z'] = standardize(df['ppl_W32'])
df['ppl_W64_z'] = standardize(df['ppl_W64'])

if 'repeat_trigram_rate' in df.columns:
    df['repeat_trigram_rate_z'] = standardize(df['repeat_trigram_rate'])
    df['distinct3_z'] = standardize(df['distinct3'])

df['score_bin'] = pd.Categorical(df['score_bin'], categories=GROUP_ORDER, ordered=True)

if 'regime' in df.columns:
    df['regime'] = pd.Categorical(df['regime'], categories=['short', 'main', 'extended'], ordered=True)

print("Standardized variables created")

## 2. Full Regression Suite (Models A, B, B', C, C')

In [ ]:
def run_full_regression_suite(df, metric):
    """Run all models including B' and C' with ppl_W64."""
    results = {}
    has_regime = 'regime' in df.columns and df['regime'].nunique() > 1
    regime_term = ' + C(regime)' if has_regime else ''
    has_rep = 'repeat_trigram_rate_z' in df.columns
    
    models = {
        'A': f'{metric} ~ C(score_bin) + token_count_z{regime_term}',
        'B': f'{metric} ~ C(score_bin) + token_count_z{regime_term} + ppl_W32_z',
        "B'": f'{metric} ~ C(score_bin) + token_count_z{regime_term} + ppl_W64_z',
    }
    
    if has_rep:
        models['C'] = f'{metric} ~ C(score_bin) + token_count_z + ppl_W32_z + repeat_trigram_rate_z + distinct3_z'
        models["C'"] = f'{metric} ~ C(score_bin) + token_count_z + ppl_W64_z + repeat_trigram_rate_z + distinct3_z'
    
    for name, formula in models.items():
        try:
            subset_cols = [metric, 'token_count_z']
            if 'ppl_W32_z' in formula:
                subset_cols.append('ppl_W32_z')
            if 'ppl_W64_z' in formula:
                subset_cols.append('ppl_W64_z')
            if 'repeat_trigram_rate_z' in formula:
                subset_cols.extend(['repeat_trigram_rate_z', 'distinct3_z'])
            
            model = smf.ols(formula, data=df.dropna(subset=subset_cols)).fit()
            results[name] = {'formula': formula, 'model': model}
        except Exception as e:
            results[name] = {'formula': formula, 'error': str(e)}
    
    return results

# Run for key metrics
all_results = {}
for metric in KEY_METRICS:
    all_results[metric] = run_full_regression_suite(df, metric)

# Display results
print("="*80)
print("REGRESSION RESULTS: GROUP EFFECTS ACROSS MODELS")
print("="*80)

for metric in KEY_METRICS:
    print(f"\n{'='*70}")
    print(f"{metric.upper()}")
    print(f"{'='*70}")
    
    results = all_results[metric]
    
    # Header
    print(f"\n{'Model':<8} {'R²':>8} {'β(mid)':>10} {'p(mid)':>10} {'β(high)':>10} {'p(high)':>10}")
    print("-"*60)
    
    for model_name in ['A', 'B', "B'", 'C', "C'"]:
        if model_name not in results:
            continue
        r = results[model_name]
        if 'error' in r:
            print(f"{model_name:<8} ERROR: {r['error'][:40]}")
            continue
        
        m = r['model']
        beta_mid = m.params.get('C(score_bin)[T.mid]', np.nan)
        p_mid = m.pvalues.get('C(score_bin)[T.mid]', np.nan)
        beta_high = m.params.get('C(score_bin)[T.high]', np.nan)
        p_high = m.pvalues.get('C(score_bin)[T.high]', np.nan)
        
        sig_mid = "*" if p_mid < 0.05 else ""
        sig_high = "*" if p_high < 0.05 else ""
        
        print(f"{model_name:<8} {m.rsquared:>8.4f} {beta_mid:>9.3f}{sig_mid} {p_mid:>10.4f} {beta_high:>9.3f}{sig_high} {p_high:>10.4f}")

## 3. Interaction Tests

Does the fluency→AUC slope differ by score bin?

In [ ]:
print("="*80)
print("INTERACTION TESTS: Does fluency effect differ by score bin?")
print("="*80)

has_regime = 'regime' in df.columns and df['regime'].nunique() > 1
regime_term = ' + C(regime)' if has_regime else ''

interaction_results = {}

for metric in KEY_METRICS:
    print(f"\n{'='*60}")
    print(f"{metric}: C(score_bin) * ppl_W32_z")
    print(f"{'='*60}")
    
    # Model with interaction
    formula_int = f'{metric} ~ C(score_bin) * ppl_W32_z + token_count_z{regime_term}'
    
    try:
        model_int = smf.ols(formula_int, data=df.dropna(subset=[metric, 'ppl_W32_z', 'token_count_z'])).fit()
        interaction_results[metric] = {'model': model_int, 'formula': formula_int}
        
        print(f"\nFormula: {formula_int}")
        print(f"R²: {model_int.rsquared:.4f}, n={int(model_int.nobs)}")
        
        print("\nInteraction terms:")
        for param, coef in model_int.params.items():
            if ':' in param:  # Interaction terms
                pval = model_int.pvalues[param]
                sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
                print(f"  {param}: {coef:+.4f} (p={pval:.4f}) {sig}")
        
        # Test if interactions are jointly significant
        interaction_terms = [p for p in model_int.params.index if ':' in p]
        if interaction_terms:
            # F-test for joint significance
            r_matrix = np.zeros((len(interaction_terms), len(model_int.params)))
            for i, term in enumerate(interaction_terms):
                r_matrix[i, list(model_int.params.index).index(term)] = 1
            f_test = model_int.f_test(r_matrix)
            print(f"\nJoint F-test for interactions: F={f_test.fvalue[0][0]:.3f}, p={f_test.pvalue:.4f}")
            
    except Exception as e:
        print(f"ERROR: {e}")
        interaction_results[metric] = {'error': str(e)}

## 4. Visualizations: The Story

In [ ]:
# Figure 1: Key scatter plots showing effect collapse
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Helper for regression line
def add_regression_line(ax, x, y, color='black', linestyle='--', label=None):
    valid = ~(np.isnan(x) | np.isnan(y))
    if valid.sum() > 2:
        z = np.polyfit(x[valid], y[valid], 1)
        p = np.poly1d(z)
        x_line = np.linspace(x[valid].min(), x[valid].max(), 100)
        ax.plot(x_line, p(x_line), color=color, linestyle=linestyle, linewidth=2, label=label, alpha=0.8)
        return z[0]  # slope
    return None

# 1a: auc_128 vs ppl_W32 - THE KEY PLOT
ax = axes[0, 0]
for group in GROUP_ORDER:
    g_df = df[df['score_bin'] == group]
    ax.scatter(g_df['ppl_W32'], g_df['auc_128'], c=COLORS[group], label=group, alpha=0.6, s=50, edgecolors='white', linewidth=0.5)

# Single global regression line
slope = add_regression_line(ax, df['ppl_W32'].values, df['auc_128'].values, color='black', label=f'Overall (all groups)')
ax.set_xlabel('Baseline Perplexity (ppl_W32)', fontsize=12)
ax.set_ylabel('AUC_128', fontsize=12)
ax.set_title('AUC vs Baseline Fluency\n(Group separation collapses on fluency)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 1b: log_slope_128 vs ppl_W32
ax = axes[0, 1]
for group in GROUP_ORDER:
    g_df = df[df['score_bin'] == group]
    ax.scatter(g_df['ppl_W32'], g_df['log_slope_128'], c=COLORS[group], label=group, alpha=0.6, s=50, edgecolors='white', linewidth=0.5)

add_regression_line(ax, df['ppl_W32'].values, df['log_slope_128'].values, color='black', label='Overall')
ax.set_xlabel('Baseline Perplexity (ppl_W32)', fontsize=12)
ax.set_ylabel('Log Slope_128', fontsize=12)
ax.set_title('Log Slope vs Baseline Fluency', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 1c: Violin plot of ppl_W32 by group (shows fluency differs by group)
ax = axes[1, 0]
sns.violinplot(data=df, x='score_bin', y='ppl_W32', order=GROUP_ORDER, palette=COLORS, ax=ax, inner='box')
ax.set_xlabel('Score Bin', fontsize=12)
ax.set_ylabel('Baseline Perplexity (ppl_W32)', fontsize=12)
ax.set_title('Baseline Fluency by Score Bin\n(Low-scoring essays have HIGHER perplexity)', fontsize=12, fontweight='bold')

# Add means
for i, group in enumerate(GROUP_ORDER):
    mean_val = df[df['score_bin'] == group]['ppl_W32'].mean()
    ax.annotate(f'{mean_val:.1f}', xy=(i, mean_val), ha='center', va='bottom', fontsize=10, fontweight='bold')

# 1d: Violin plot of auc_128 by group
ax = axes[1, 1]
sns.violinplot(data=df, x='score_bin', y='auc_128', order=GROUP_ORDER, palette=COLORS, ax=ax, inner='box')
ax.set_xlabel('Score Bin', fontsize=12)
ax.set_ylabel('AUC_128', fontsize=12)
ax.set_title('AUC by Score Bin\n(Apparent difference driven by fluency)', fontsize=12, fontweight='bold')

for i, group in enumerate(GROUP_ORDER):
    mean_val = df[df['score_bin'] == group]['auc_128'].mean()
    ax.annotate(f'{mean_val:.1f}', xy=(i, mean_val), ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle(f'The Story: Score-bin differences explained by baseline fluency ({EXPERIMENT})', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / EXPERIMENT / 'plots_story.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2: Coefficient comparison across models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, metric in enumerate(KEY_METRICS):
    ax = axes[idx]
    results = all_results[metric]
    
    models = ['A', 'B', "B'", 'C', "C'"]
    models = [m for m in models if m in results and 'model' in results[m]]
    
    x = np.arange(len(models))
    width = 0.35
    
    beta_mid = [results[m]['model'].params.get('C(score_bin)[T.mid]', np.nan) for m in models]
    beta_high = [results[m]['model'].params.get('C(score_bin)[T.high]', np.nan) for m in models]
    
    bars1 = ax.bar(x - width/2, beta_mid, width, label='mid vs low', color=COLORS['mid'], alpha=0.8)
    bars2 = ax.bar(x + width/2, beta_high, width, label='high vs low', color=COLORS['high'], alpha=0.8)
    
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.set_xlabel('Model', fontsize=11)
    ax.set_ylabel('Coefficient (β)', fontsize=11)
    ax.set_title(f'{metric}: Group effects across models', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add significance markers
    for i, m in enumerate(models):
        p_mid = results[m]['model'].pvalues.get('C(score_bin)[T.mid]', 1)
        p_high = results[m]['model'].pvalues.get('C(score_bin)[T.high]', 1)
        if p_mid < 0.05:
            ax.annotate('*', xy=(i - width/2, beta_mid[i]), ha='center', va='bottom', fontsize=14, fontweight='bold')
        if p_high < 0.05:
            ax.annotate('*', xy=(i + width/2, beta_high[i]), ha='center', va='bottom', fontsize=14, fontweight='bold')

plt.suptitle('Group effects shrink/vanish when controlling for fluency', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE_DIR / EXPERIMENT / 'plots_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Repetition Metrics Check

In [ ]:
# Do repetition metrics differ by score bin?
print("="*80)
print("REPETITION METRICS BY SCORE BIN")
print("="*80)

rep_metrics = ['ttr', 'repeat_bigram_rate', 'repeat_trigram_rate', 'distinct2', 'distinct3']
rep_metrics = [m for m in rep_metrics if m in df.columns]

if rep_metrics:
    print(f"\n{'Metric':<25} {'F-stat':>10} {'p-value':>12} {'low':>10} {'mid':>10} {'high':>10}")
    print("-"*80)
    
    for metric in rep_metrics:
        groups = [df[df['score_bin'] == g][metric].dropna() for g in GROUP_ORDER]
        if all(len(g) > 2 for g in groups):
            f_stat, p_val = stats.f_oneway(*groups)
            means = [g.mean() for g in groups]
            sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
            print(f"{metric:<25} {f_stat:>10.2f} {p_val:>11.4f}{sig} {means[0]:>10.4f} {means[1]:>10.4f} {means[2]:>10.4f}")
else:
    print("No repetition metrics available")

## 6. Generate Findings Summary

In [ ]:
# Generate findings paragraph
print("="*80)
print("FINDINGS SUMMARY")
print("="*80)

# Gather key statistics
auc_results = all_results['auc_128']
log_slope_results = all_results['log_slope_128']

# Model A p-values
p_auc_A = auc_results['A']['model'].pvalues.get('C(score_bin)[T.high]', 1) if 'model' in auc_results['A'] else 1
p_slope_A = log_slope_results['A']['model'].pvalues.get('C(score_bin)[T.high]', 1) if 'model' in log_slope_results['A'] else 1

# Model B p-values
p_auc_B = auc_results['B']['model'].pvalues.get('C(score_bin)[T.high]', 1) if 'model' in auc_results['B'] else 1
p_slope_B = log_slope_results['B']['model'].pvalues.get('C(score_bin)[T.high]', 1) if 'model' in log_slope_results['B'] else 1

# Interaction p-values
int_auc = interaction_results.get('auc_128', {})
int_slope = interaction_results.get('log_slope_128', {})

findings = f"""
FINDINGS ({EXPERIMENT} cohort, n={len(df)})
{'-'*60}

Score-bin differences in memory-curve shape (AUC, log_slope) are 
{'significant' if p_auc_A < 0.05 or p_slope_A < 0.05 else 'not significant'} when controlling for length alone 
(AUC: p={'<0.001' if p_auc_A < 0.001 else f'{p_auc_A:.3f}'}, log_slope: p={'<0.001' if p_slope_A < 0.001 else f'{p_slope_A:.3f}'}), 
but {'vanish' if p_auc_B > 0.05 and p_slope_B > 0.05 else 'persist'} when controlling for baseline perplexity 
(AUC: p={p_auc_B:.3f}, log_slope: p={p_slope_B:.3f}).

The fluency→AUC relationship does not differ significantly by score bin 
(interaction test p={'>.05' if 'model' not in int_auc or all(int_auc['model'].pvalues.get(p, 1) > 0.05 for p in int_auc['model'].params.index if ':' in p) else '<.05'}).
"""

# Add repetition finding
if rep_metrics:
    rep_significant = []
    for metric in rep_metrics:
        groups = [df[df['score_bin'] == g][metric].dropna() for g in GROUP_ORDER]
        if all(len(g) > 2 for g in groups):
            _, p = stats.f_oneway(*groups)
            if p < 0.05:
                rep_significant.append(metric)
    
    if rep_significant:
        findings += f"\nRepetition metrics that differ by score bin: {', '.join(rep_significant)}.\n"
    else:
        findings += "\nRepetition metrics do not differ significantly by score bin.\n"

findings += """
CONCLUSION: Apparent group differences in curve shape are largely 
explained by baseline fluency/difficulty rather than differential 
long-range dependence. Low-scoring essays have higher perplexity 
(are harder for the LM to predict), which mechanically produces 
larger AUC/slope values. Once fluency is controlled, the score-bin 
effect disappears.
"""

print(findings)

# Save findings
with open(BASE_DIR / EXPERIMENT / 'findings_summary.txt', 'w') as f:
    f.write(findings)

## 7. Save All Results

In [ ]:
output_dir = BASE_DIR / EXPERIMENT

# Save updated essay-level results
df.to_csv(output_dir / 'essay_level_results.csv', index=False)

# Save regression summaries
with open(output_dir / 'regression_summary_full.txt', 'w') as f:
    f.write("FULL REGRESSION SUITE: Models A, B, B', C, C'\n")
    f.write("="*80 + "\n\n")
    
    for metric in KEY_METRICS:
        f.write(f"\n{'#'*80}\n")
        f.write(f"# {metric}\n")
        f.write(f"{'#'*80}\n\n")
        
        results = all_results[metric]
        for model_name in ['A', 'B', "B'", 'C', "C'"]:
            if model_name not in results:
                continue
            r = results[model_name]
            f.write(f"\n--- Model {model_name} ---\n")
            f.write(f"Formula: {r['formula']}\n")
            if 'model' in r:
                f.write(r['model'].summary().as_text())
            else:
                f.write(f"ERROR: {r.get('error', 'Unknown')}\n")
            f.write("\n")

# Save interaction test results
with open(output_dir / 'interaction_tests.txt', 'w') as f:
    f.write("INTERACTION TESTS: Does fluency effect differ by score bin?\n")
    f.write("="*80 + "\n\n")
    
    for metric in KEY_METRICS:
        f.write(f"\n{'='*60}\n")
        f.write(f"{metric}\n")
        f.write(f"{'='*60}\n")
        
        r = interaction_results.get(metric, {})
        if 'model' in r:
            f.write(f"Formula: {r['formula']}\n")
            f.write(r['model'].summary().as_text())
        else:
            f.write(f"ERROR: {r.get('error', 'Not run')}\n")

print(f"\nSaved to {output_dir}/")
print("  - essay_level_results.csv")
print("  - regression_summary_full.txt")
print("  - interaction_tests.txt")
print("  - findings_summary.txt")
print("  - plots_story.png")
print("  - plots_coefficients.png")